# Libraries

In [1]:
import warnings
from pathlib import Path

from tqdm import tqdm

from matplotlib import pyplot as plt
import numpy as np
import seaborn as sn
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance
import shap
from boruta import BorutaPy
import neurokit2 as nk
from helper_code import *

/home/matteo/Documenti/VSCODE/ECG-Chagas/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Settings

In [2]:
%load_ext autoreload

%autoreload 2

# Control figure size
%matplotlib inline
figsize=(14, 4)

warnings.filterwarnings('ignore')

# Detection of Chagas Disease from the ECG: PhysioNet Challenge 2025

## Introduction

Chagas disease is a potentially life-threatening illness caused by the protozoan parasite Trypanosoma cruzi (T. cruzi). It is found mainly in endemic areas of 21 countries in Latin America, where it is mostly transmitted to humans by the feces of triatomine bugs known as "kissing bugs". The disease is also known as American trypanosomiasis.

The disease is named after the Brazilian physician Carlos Chagas, who discovered it in 1909. The disease can be acute or chronic, but the chronic form is the most common and usually develops 10 to 30 years after the initial infection. The acute phase of the disease is usually mild or asymptomatic, but the chronic phase can cause serious heart and digestive problems.

The disease is diagnosed by detecting the parasite in the blood or by serological tests. The electrocardiogram (ECG) is an important tool for diagnosing Chagas disease, as it can show changes in the heart's electrical activity that are characteristic of the disease.

The goal of this challenge is to develop algorithms that can automatically detect Chagas disease from ECG signals. The challenge is based on a dataset of ECG signals from patients with Chagas disease and healthy controls. The dataset is provided by the PhysioNet Challenge 2025.

## EDA

In [3]:
exams = pd.read_csv('code15_hdf5/exams.csv')
all_labels = pd.read_csv('code15_hdf5/code15_chagas_labels.csv')

In [4]:
train_data_folder = Path("code15_wfdb")
holdout_data_folder = Path("holdout_data")

train_records = find_records(train_data_folder.absolute().__str__())
holdout_records = find_records(holdout_data_folder.absolute().__str__())

all_records = train_records + holdout_records

In [ ]:
exams[exams.isin([int(x) for x in all_records]).any(axis=1)].merge(all_labels, on='exam_id').chagas.value_counts(normalize=True)

chagas
False    49199
True      6559
Name: count, dtype: int64

In [5]:
def get_random_patient(random: bool = True, with_disease=True):
    if random:
        random_id = np.random.randint(0, len(train_records))
    else:
        random_id = 34
    record = str(train_data_folder.absolute()) + "/" + train_records[random_id]
    header = load_header(record)
    signals, fields = load_signals(record)
    frequency = get_sampling_frequency(header)
    label = get_label(header)

    if label == 0 and with_disease:
        return get_random_patient(with_disease)
    elif label == 1 and not with_disease:
        return get_random_patient(with_disease)
    else:
        return record, train_records[random_id], label, signals, frequency

In [ ]:
record, exam_id, label, signals, frequency = get_random_patient(random=False, with_disease=False)
print(f"Record: {record}, Exam ID: {exam_id}, Label: {label}")
exams[exams['exam_id'] == int(exam_id)]

In [ ]:
wfdb.plot_wfdb(record=wfdb.rdrecord(record, physical=False), figsize=(20, 10), time_units='samples')

---

In [8]:
# Define a new function
def my_processing(ecg_signals: np.ndarray, channel: int, frequency: int):
    # Do processing
    ecg_cleaned = nk.ecg_clean(ecg_signals[:, channel], sampling_rate=frequency, method="biosppy")
    peaks, info = nk.ecg_peaks(ecg_cleaned, sampling_rate=frequency)
    hrv_time = nk.hrv_time(peaks, sampling_rate=frequency, show=False)
    waves, waves_signals = nk.ecg_delineate(ecg_cleaned, info, sampling_rate=frequency, method="dwt", check=True, show=False, show_type='all')

    phases = nk.ecg_phase(ecg_cleaned, peaks, sampling_rate=frequency, delineate_info=waves_signals)
    rate = nk.ecg_rate(info, sampling_rate=frequency, desired_length=len(ecg_cleaned))
    quality = nk.ecg_quality(ecg_cleaned, sampling_rate=frequency)

    # Prepare output
    signals = pd.DataFrame({"ECG_Raw": ecg_signals[:, channel],
                            "ECG_Clean": ecg_cleaned,
                            "ECG_Rate": rate,
                            "ECG_Quality": quality})
    signals = pd.concat([signals, peaks, waves, phases], axis=1)

    # Create info dict
    info = info
    info["sampling_rate"] = frequency

    waves_signals['ECG_R_Peaks'] = info['ECG_R_Peaks']
    
    return signals, info, waves_signals

In [9]:
processed_signals, info, waves_signals = my_processing(signals, 0, frequency)

In [10]:
def check_interval(waves_signals):
    df_waves = pd.DataFrame(waves_signals)
    df_waves.drop(columns=['ECG_R_Onsets', 'ECG_R_Offsets'], inplace=True)
    correct_indices = ['ECG_P_Onsets', 'ECG_P_Peaks', 'ECG_P_Offsets',
       'ECG_Q_Peaks', 'ECG_R_Peaks', 'ECG_S_Peaks', 'ECG_T_Onsets', 'ECG_T_Peaks', 'ECG_T_Offsets', ]
    rows = df_waves.shape[0]
    mask = np.zeros(rows)
    for i in range(1, rows):
        start_interval = df_waves.loc[i-1, 'ECG_R_Peaks']
        end_interval = df_waves.loc[i, 'ECG_R_Peaks']
        sliced_df = df_waves[df_waves.isin([start_interval, end_interval]).any(axis=1)]
        # if the sliced df has no missing values, then the interval is correct and the mask should be 1
        first_condition = sliced_df[correct_indices].isnull().sum().sum() == 0
        second_condition = correct_indices == sliced_df.iloc[0].sort_values(ascending=True).index.tolist()
        if first_condition and second_condition:
            mask[i] = 1
    return df_waves[mask == 1]

In [11]:
def ecg_signal_features(row, frequency: int = 400):
    ECG_P_Peaks, ECG_P_Onsets, ECG_P_Offsets, ECG_Q_Peaks, ECG_S_Peaks, ECG_T_Peaks, ECG_T_Onsets, ECG_T_Offsets, ECG_R_Peaks = row
    # Compute the P wave duration
    P_wave_duration = (ECG_P_Offsets - ECG_P_Onsets) / frequency
    # Compute the PR interval
    PR_interval = (ECG_Q_Peaks - ECG_P_Onsets) / frequency
    # Compute the PR segment
    PR_segment = (ECG_Q_Peaks - ECG_P_Offsets) / frequency
    # Compute the QRS duration
    QRS_duration = (ECG_S_Peaks - ECG_Q_Peaks) / frequency
    # Compute the QT interval
    QT_interval = (ECG_T_Offsets - ECG_Q_Peaks) / frequency
    # Compute the ST segment
    ST_segment = (ECG_T_Onsets - ECG_S_Peaks) / frequency

    return {
        "P_wave_duration": P_wave_duration,
        "PR_interval": PR_interval,
        "PR_segment": PR_segment,
        "QRS_duration": QRS_duration,
        "QT_interval": QT_interval,
        "ST_segment": ST_segment
    }

In [12]:
def st_slope(signal_df, s_peak, t_onset):
    return (signal_df.iloc[t_onset]['ECG_Clean'] - signal_df.iloc[s_peak]['ECG_Clean']) / (t_onset - s_peak)

In [13]:
correct_waves = check_interval(waves_signals)

In [14]:
correct_waves[["P_wave_duration","PR_interval","PR_segment","QRS_duration","QT_interval","ST_segment"]] = correct_waves.apply(lambda x: ecg_signal_features(x, frequency), axis=1, result_type='expand')

In [15]:
correct_waves['ST_slope'] = correct_waves.apply(lambda x: st_slope(processed_signals, int(x['ECG_S_Peaks']), int(x['ECG_T_Onsets'])), axis=1)

In [ ]:
correct_waves

In [ ]:
error_records = []
channel = 1
hand_features = pd.DataFrame()
for i in tqdm(range(len(train_records))):
    record = str(train_data_folder.absolute()) + "/" + train_records[i]
    header = load_header(record)
    signals, fields = load_signals(record)
    frequency = get_sampling_frequency(header)

    try:
        processed_signals, _, waves_signals = my_processing(signals, channel, frequency)
    except:
        error_records.append(train_records[i])
        continue
    correct_waves = check_interval(waves_signals)
    if correct_waves.shape[0] == 0:
        error_records.append(train_records[i])
        continue
    correct_waves[["P_wave_duration","PR_interval","PR_segment","QRS_duration","QT_interval","ST_segment"]] = correct_waves.apply(lambda x: ecg_signal_features(x, frequency), axis=1, result_type='expand')
    correct_waves['ST_slope'] = correct_waves.apply(lambda x: st_slope(processed_signals, int(x['ECG_S_Peaks']), int(x['ECG_T_Onsets'])), axis=1)
    correct_waves['record'] = train_records[i]
    hand_features = pd.concat([hand_features, correct_waves])

In [22]:
# save correct waves dataframe to a csv file
hand_features.to_csv('hand_features.csv', index=False)

# save error records to a csv file
pd.DataFrame(error_records).to_csv('error_records.csv', index=False)

---

In [66]:
test = pd.read_csv('hand_features.csv')

In [67]:
what_i_want = test.iloc[:, 9:].groupby('record').agg(['mean', 'std', 'min', 'max']).reset_index()
what_i_want.columns = ['_'.join(col).strip() for col in what_i_want.columns.values]
what_i_want.rename(columns={'record_': 'exam_id'}, inplace=True)

In [ ]:
what_i_want

In [ ]:
# Get the labels for each record id
extracted_labels = {}
for i in tqdm(range(len(train_records))):
    record = str(train_data_folder.absolute()) + "/" + train_records[i]
    header = load_header(record)
    label = get_label(header)
    extracted_labels[int(train_records[i])] = label

In [70]:
labels_df = pd.DataFrame(extracted_labels.items(), columns=['exam_id', 'label'])

In [71]:
what_i_want = what_i_want.merge(labels_df, on='exam_id')

In [72]:
final_version = exams[exams['exam_id'].isin(test['record'])].merge(what_i_want, on='exam_id', how='left')
final_version = final_version.drop(columns=['exam_id', 'patient_id', 'nn_predicted_age', '1dAVb', 'RBBB',
       'LBBB', 'SB', 'ST', 'AF', 'patient_id', 'death', 'timey', 'normal_ecg',
       'trace_file'])

In [73]:
final_version = final_version.fillna(value=0)

---

In [74]:
df = final_version

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.label.value_counts(normalize=True)

In [77]:
df['is_male'] = df['is_male'].astype(int)

In [ ]:
num_cols = [c for c in df.columns[:-1] if df[c].nunique() > 2]
cat_cols = [c for c in df.columns[:-1] if df[c].nunique() == 2]
print(f'Numeric: {num_cols}\nTotal: {len(num_cols)}')
print(f'Categorical: {cat_cols}\nTotal: {len(cat_cols)}')

In [ ]:
_, axes = plt.subplots(nrows=4, ncols=int(np.ceil(len(num_cols)//4)), figsize=(20, 10))
for ax, cname in zip(axes.ravel(), num_cols):
    df.hist(cname, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
_, axes = plt.subplots(nrows=1, ncols=1, figsize=(20, 10))
for cname in cat_cols:
    df[cname].value_counts().plot(kind='bar', ax=axes)

In [ ]:
df.hist('label', bins=2, figsize=figsize)
plt.tight_layout()

In [ ]:
_, axes = plt.subplots(nrows=2, ncols=int(np.ceil(len(cat_cols)//2)), figsize=figsize)
for ax, cname in zip(axes.ravel(), cat_cols):
    df.groupby(cname)['label'].mean().plot.bar(ax=ax)
plt.tight_layout()

In [ ]:
_, axes = plt.subplots(nrows=4, ncols=int(np.ceil(len(num_cols)//4)), figsize=(20,10))
for ax, cname in zip(axes.ravel(), num_cols):
    bin_size = (df[cname].max() - df[cname].min()) / 20
    df['label'].groupby(df[cname] // bin_size).mean().plot.bar(ax=ax)
plt.tight_layout()

In [ ]:
plt.figure(figsize=(27,12))
sn.heatmap(df.corr(method='pearson', numeric_only=True), annot=True, vmin=-1, vmax=1, cmap='RdBu')
plt.tight_layout()

In [148]:
from imblearn.over_sampling import SMOTE

In [162]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

clean_df = pd.concat([df[df['label'] == 1], df[df['label'] == 0].sample(n=400)], axis=0)

X, y = clean_df.drop(columns='label'), clean_df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

In [ ]:
y.value_counts()

In [165]:
base_est = LogisticRegression(penalty='l1', solver='saga',)
param_grid={'C': 1. / np.linspace(1e-1, 1e4, 100)}
gscv = GridSearchCV(base_est, param_grid=param_grid, scoring='roc_auc')
gscv.fit(X_train, y_train)
lr, lr_params = gscv.best_estimator_, gscv.best_params_

In [ ]:
lr_score_cv, lr_score_test = gscv.best_score_, roc_auc_score(y_test, lr.predict_proba(X_test)[:, 1])
print(f'AUC score for C={lr_params["C"]:.2f}: {lr_score_cv:.2f} (cross-validation), {lr_score_test:.2f} (test)')

In [ ]:
from sklearn.metrics import f1_score, confusion_matrix

cm = confusion_matrix(y_test, lr.predict(X_test), labels=[0, 1], normalize='all')
print(f1_score(y_test, lr.predict(X_test)))
print(f'Confusion matrix:\n{cm}')
sn.heatmap(cm, annot=True, cmap='cividis')

In [ ]:
pd.Series( lr.predict_proba(X_test)[:, 1]).describe()

In [ ]:
lr_coefs = pd.Series(index=X.columns, data=lr.coef_[0])
lr_coefs.plot(kind='bar', figsize=figsize)

In [ ]:
import xgboost
base_est = xgboost.XGBClassifier(tree_method='hist', importance_type='total_gain')
param_grid={'max_depth': [2, 3, 4], 'n_estimators': list(range(20, 41, 5)), 'reg_lambda': np.linspace(0, 500, 6)}
gscv = GridSearchCV(base_est, param_grid=param_grid)
gscv.fit(X_train, y_train)
xbm, xbm_params = gscv.best_estimator_, gscv.best_params_

In [ ]:
plt.figure(figsize=figsize)
xgboost.plot_tree(xbm, ax=plt.gca(), num_trees=0)

In [ ]:
xbm_score_cv, xbm_score_test = gscv.best_score_, roc_auc_score(y_test, xbm.predict(X_test))
print(f'AUC score for {xbm_params}: {xbm_score_cv:.2f} (cross-val.), {xbm_score_test:.2f} (test)')

In [ ]:
xbm_imp = pd.Series(index=X.columns, data=xbm.feature_importances_)
xbm_imp.plot(kind='bar', figsize=figsize)

In [ ]:
_, axes = plt.subplots(nrows=2, ncols=3, figsize=figsize)
for ax, imp_type in zip(axes.ravel(), ['weight', 'gain', 'cover', 'total_gain', 'total_cover']):
    pd.Series(xbm.get_booster().get_score(importance_type=imp_type)).plot.bar(ax=ax, title=imp_type)
plt.tight_layout()

In [ ]:
import util

r_train = permutation_importance(xbm, X_train, y_train, n_repeats=30, random_state=42)
xbm_p_imp = pd.Series(index=X.columns, data=r_train.importances_mean)
util.plot_bars(xbm_p_imp, figsize=figsize, std=r_train.importances_std, title='Permutation Feature Importance (train)')

In [ ]:
r_test = permutation_importance(xbm, X_test, y_test, n_repeats=30, random_state=42)
xbm_p_imp = pd.Series(index=X.columns, data=r_test.importances_mean)
util.plot_bars(xbm_p_imp, figsize=figsize, std=r_test.importances_std, title='Permutation Feature Importance (test)')

In [ ]:
f = lambda x: xbm.predict_proba(x)[:,1]
explainer = shap.KernelExplainer(f, shap.sample(X_train, 100), link='logit')
shap_values = explainer(X_test)

In [ ]:
bfs = BorutaPy(xbm, n_estimators='auto', max_iter=100, verbose=0, random_state=42)
bfs.fit(X=X, y=y)

In [ ]:
print('Confirmed important:', X.columns[bfs.ranking_ == 1].values)
print('Uncomfirmed:', X.columns[bfs.ranking_ == 2].values)
print('Confirmed unimportant:', X.columns[bfs.ranking_ > 2].values)